# kubera-quant Quickstart

This notebook demonstrates the core workflow:
1. Load OHLCV data from local Parquet cache
2. Apply a strategy to generate signals
3. Run backtest and view results

In [ ]:
import pandas as pd
from quant.core.data.cache import ParquetCache
from quant.core.strategy import get_strategy, STRATEGIES
from quant.core.backtest.engine import run_backtest
from quant.core.backtest.report import print_report

## 1. Available strategies

In [ ]:
for name, s in STRATEGIES.items():
    print(f"{name:20s} {s.description}")

## 2. Load data from cache

First, collect data and sync to local cache via CLI:
```bash
kubera-quant collect krx --symbols 005930 --start 2020-01-01
kubera-quant sync krx
```

In [ ]:
cache = ParquetCache()
df = cache.read(market="krx", symbol="005930")
print(f"Loaded {len(df)} rows")
df.tail()

## 3. Generate signals with SMA crossover

In [ ]:
strategy = get_strategy("sma_cross")
signal = strategy.generate_signal(df, short=5, long=20)

print(f"Buy signals:  {(signal == 1.0).sum()}")
print(f"Sell signals: {(signal == -1.0).sum()}")
print(f"Hold:         {(signal == 0.0).sum()}")

## 4. Run backtest

In [ ]:
result = run_backtest(df, signal, init_cash=10_000_000)
print_report(result, strategy_name="sma_cross", symbol="005930")

## 5. Visualize with vectorbt

The `BacktestResult.portfolio` is a raw vectorbt Portfolio object.

In [ ]:
# Equity curve
result.portfolio.value().plot(title="Portfolio Value")

In [ ]:
# Drawdown
result.portfolio.drawdowns.plot()

## 6. Compare strategies

In [ ]:
results = {}
for name in ["sma_cross", "momentum"]:
    s = get_strategy(name)
    sig = s.generate_signal(df, short=5, long=20, lookback=20)
    r = run_backtest(df, sig)
    results[name] = r
    print(f"{name:20s} return={r.total_return:+.2%}  sharpe={r.sharpe_ratio:.4f}  mdd={r.max_drawdown:.2%}")